# Datenquelle US Census Bureau: TIGER/Line und ACS

Diese Quelle liefert nicht Ereignisse, sondern **Referenzdaten**: die Geometrie
der Census Tracts und ihre sozio-ökonomischen Kennzahlen. Sie ist der Grund,
warum sich Chicago und Detroit überhaupt vergleichen lassen.

**Pipeline-Stufe:** Quellendokumentation. Die Ingestion selbst liegt in
`census_pipeline` (`download_tiger.py`, `download_acs.py`).

| | TIGER/Line | ACS 5-Jahres-Schätzungen |
| --- | --- | --- |
| Inhalt | Grenzen der Tracts und der Orte als Polygone | Einwohner, Einkommen, Armut je Tract |
| Herausgeber | US Census Bureau | US Census Bureau |
| Zugriff | ZIP-Datei über HTTPS | JSON-API mit Schlüssel |
| Aenderungsverhalten | eine Vintage pro Jahr, danach eingefroren | dito |
| Schlüssel | `GEOID` | `GEOID` (aus state+county+tract) |

TIGER liefert dabei **zwei Auslieferungen**: `tract` für das Vergleichsraster
und `place` für die Stadtgrenzen. Beide brauchen einen eigenen Lauf.

## 1. Warum diese Quelle

Chicago und Detroit liefern beide Koordinaten, aber ihre eigenen Raumeinheiten
sind **nicht** vergleichbar:

| Stadt | Eigene Einheiten | Vergleichbar? |
| --- | --- | --- |
| Chicago | `community_area`, `ward`, `district`, `beat` | nur innerhalb Chicagos |
| Detroit | Precincts, Neighborhoods | nur innerhalb Detroits |
| beide | **Census Tract** | ja, bundesweit einheitlich |

Ein Census Tract umfasst typischerweise 1'200 bis 8'000 Einwohner und ist so
geschnitten, dass er über die Jahre halbwegs stabil bleibt. Er ist die
kleinste Einheit, für die das Census Bureau verlässliche sozio-ökonomische
Werte veröffentlicht.

Damit werden zwei Dinge möglich, die vorher nicht gingen:

1. **Raten statt absoluter Zahlen.** 500 Fälle in einem Tract mit 800
   Einwohnern bedeuten etwas anderes als 500 Fälle in einem Tract mit 6'000.
2. **Ein gemeinsames Raster.** Dieselbe Flächendefinition in beiden Städten,
   statt Chicagos 77 community areas gegen Detroits Precincts zu stellen.

Dafür braucht es allerdings eine zweite Geometrie: Ein Tract sagt nicht, zu
welcher Stadt er gehört. Die Pipeline lädt ganze Bundesstaaten, und ein County
ist weit grösser als die Stadt darin - Cook County zählt 5,18 Millionen
Einwohner, Chicago rund 2,7 Millionen. Deshalb kommt die Auslieferung `place`
dazu; sie beantwortet die Frage, wo die Stadt aufhört, und liefert damit erst
den Nenner jeder Rate.

## 2. Was eine Zeile bedeutet

**TIGER:** eine Zeile = ein Tract = ein Polygon. Die Geometrie beschreibt die
Fläche, die Attribute identifizieren sie.

**ACS:** eine Zeile = ein Tract = seine geschätzten Kennzahlen. Wichtig:
**Schätzungen**, keine Zählungen. Der ACS befragt eine Stichprobe; die
5-Jahres-Variante bündelt 60 Monate, damit auch kleine Tracts belastbare Werte
bekommen. Der Preis dafür: Die Werte beziehen sich auf einen Zeitraum, nicht
auf einen Stichtag.

Beide Zeilen treffen sich über `GEOID` - 11 Stellen, zusammengesetzt aus
Bundesstaat (2), County (3) und Tract (6):

```text
26  163   000100     ->  26163000100
MI  Wayne Tract 1
```

## 3. Vintages: welches Jahr gilt

Eine Vintage erscheint einmal jährlich und ist danach **eingefroren**. Das ist
der wichtigste Unterschied zu den Falldaten und der Grund, warum die Ingestion
ohne Wasserzeichen und ohne Delta auskommt.

**Die Falle:** Tract-Grenzen ändern sich mit jedem Zensus. Die Grenzen des
Zensus 2020 gelten ab Vintage 2020; davor gelten die Grenzen von 2010. Wer
Fälle von 2001 bis 2019 verortet, braucht deshalb **zusätzlich** eine
ältere Vintage - und darf die Ergebnisse nicht über den Bruch hinweg
aneinanderreihen, ohne das kenntlich zu machen.

Deshalb steht die Vintage im Pfad und ist nirgends fest verdrahtet:

```text
data/bronze/dataset=tiger_tract/vintage=2024/state=26/run_.../part-0001.parquet
data/bronze/dataset=tiger_place/vintage=2024/state=26/run_.../part-0001.parquet
data/bronze/dataset=acs5_tract/vintage=2024/state=26/run_.../part-0001.parquet
```

## 4. Technischer Zugriff

### TIGER/Line

Eine ZIP-Datei je Bundesstaat und Vintage, über ein festes Adressmuster. Kein
Schlüssel, keine Paginierung, keine Rate-Limits:

```text
https://www2.census.gov/geo/tiger/TIGER<vintage>/TRACT/tl_<vintage>_<fips>_tract.zip
```

`<fips>` ist die Nummer des Bundesstaats: **17** = Illinois (Chicago),
**26** = Michigan (Detroit).

### ACS

Eine JSON-API, die einen kostenlosen Schlüssel verlangt
(`api.census.gov/data/key_signup.html`). Er liegt als `CENSUS_API_KEY` in
`.env`:

```text
https://api.census.gov/data/<vintage>/acs/acs5?get=...&for=tract:*&in=state:26&key=...
```

Anders als bei Socrata gehört der Schlüssel hier in die **Abfrage**, nicht in
einen Header. Die Ingestion legt deshalb im Manifest bewusst die URL **ohne**
Schlüssel ab.

## 5. Feststellungen aus der Erkundung

Die folgenden Punkte sind nicht aus der Dokumentation abgeschrieben, sondern an
der Quelle geprüft. Sie sind der Grund für mehrere Designentscheidungen der
Ingestion.

### 5.1 Ohne Schlüssel antwortet die ACS-API mit HTTP 200

Das ist die unangenehmste Eigenheit: Fehlt der Schlüssel, kommt **kein**
401 oder 403, sondern **HTTP 200 mit einer HTML-Seite** "Missing Key". Code,
der nur den Statuscode prüft, hält das für einen Erfolg und schreibt einen
leeren Lauf. `download_acs.parse_payload()` prüft deshalb die Antwort selbst.

### 5.2 Das TIGER-Schema driftet zwischen den Vintages

Vintage 2024 hat die Spalte `GEOIDFQ`, Vintage 2020 nicht - sonst sind die
Spalten identisch. Genau dafür gibt es das gepinnte Schema plus `_extra`: Die
zusätzliche Spalte landet unverändert als JSON in `_extra`, statt das Schema
von Lauf zu Lauf wandern zu lassen.

### 5.3 TIGER liefert NAD83, die Falldaten WGS84 - und niemand rechnet um

Die Shapefiles stehen in **EPSG:4269** (NAD83), die Koordinaten aus Chicago und
Detroit in **EPSG:4326** (WGS84). In den kontinentalen USA liegen die beiden
Bezugssysteme ein bis zwei Meter auseinander.

Umgerechnet wird **nicht**. `build_crime_offense` joint mit
`ST_Within(ST_Point(longitude, latitude), geom)`; ein `ST_Transform` kommt in
der ganzen Pipeline nicht vor. Die beiden Systeme werden also stillschweigend
gleichgesetzt.

Lange galt das als folgenlos, weil der Versatz kleiner ist als die Genauigkeit
der Falldaten. Die Messung in 6.7 hält dem nicht stand: **rund ein Viertel** der
verorteten Fälle liegt näher als einen Meter an der Grenze genau des Tracts, dem
sie zugeordnet wurden - gut das Vierfache dessen, was eine Gleichverteilung
innerhalb der Tracts erwarten liesse (6,00 %). Für diese Fälle entscheidet ein Versatz in
der Grössenordnung des Datumsunterschieds tatsächlich über die Zuordnung.

Die Ursache liegt aber nicht im Datum, sondern in der Verortung: Beide Städte
setzen ihre Koordinaten auf Strassenachsen, und TIGER schneidet Tract-Grenzen
genau entlang derselben Strassen. Die Tract-Zuordnung dieser Fälle ist auf
Meterebene deshalb ohnehin nicht belastbar - mit Umrechnung so wenig wie ohne.
Für Auswertungen auf Tract-Ebene ist das hinnehmbar, weil Fehlzuordnungen
zwischen Nachbar-Tracts sich teilweise ausgleichen; für Aussagen über einzelne
Tract-Grenzen ist es das nicht. Die Umrechnung nachzurüsten ist damit eine
offene Entscheidung - dokumentiert, nicht erledigt.

Wie stark das wiegt, zeigt Detroit, das sein eigenes Tract mitliefert
(`census_block_2020_geoid`). Unsere Zuordnung stimmt in 83,44 % der Fälle mit
seiner überein. Von den abweichenden Fällen liegen aber **93,0 % näher als rund
neun Meter** an genau dem Tract, das Detroit nennt, und 49,4 % näher als einen
Meter. Die Abweichung ist also fast durchgehend eine Frage der Kante, nicht
eine falsche Verortung.

### 5.4 `-666666666` heisst "nicht verfügbar"

Der ACS verwendet negative Sentinel-Werte für Fälle, in denen keine
Schätzung möglich ist - etwa beim Median-Einkommen eines Tracts ohne
Einwohner. Allein in Michigan trifft das 146 Tracts. Bronze lässt den Wert
unverändert stehen; ihn als "fehlend" zu deuten, ist Aufgabe der
Silver-Schicht.

### 5.5 TIGER/Line, nicht Cartographic Boundary

Das Census Bureau bietet dieselben Flächen auch als **Cartographic Boundary
Files** an - generalisiert, kleiner, hübscher für Karten. Für
Punkt-in-Polygon sind sie ungeeignet: An generalisierten Rändern fallen Punkte
in die falsche Fläche. Für die Verortung von Fällen gilt TIGER/Line.

### 5.6 Ein ZIP beweist seine eigene Unversehrtheit

Die Quelle nennt keine Zeilenzahl, gegen die sich zählen liesse. Aber ein ZIP
führt je Mitgliedsdatei eine **CRC32**-Prüfsumme mit. Zusammen mit
`Content-Length` und den vier Pflichtdateien eines Shapefiles
(`.shp`, `.shx`, `.dbf`, `.prj`) ist das der Vollständigkeitsnachweis dieser
Quelle - das Gegenstück zur festgeschriebenen id-Liste bei Chicago.

## 6. Die Quelle ansehen

Ab hier wird geprüft statt behauptet. Zuerst die Module und ihre Konstanten -
ohne Netzzugriff.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'census_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'census_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/census_pipeline nicht gefunden')

sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

import requests
from census_pipeline import STATES, download_acs as da, download_tiger as dt

print('Bundesstaaten :', STATES)
print()
print('TIGER  Vintage:', dt.DEFAULT_VINTAGE, '| Dataset:', dt.DATASET)
print('       CRS    :', dt.EXPECTED_CRS)
print('       Pflicht:', dt.REQUIRED_MEMBERS)
print()
print('ACS    Vintage:', da.DEFAULT_VINTAGE, '| Dataset:', da.DATASET)
print('       Schlüssel aus:', da.KEY_VARIABLE)
for code_, label in da.VARIABLES.items():
    print(f'       {code_:14} {label}')

### 6.1 Welche Vintages es gibt

Das Verzeichnis der Quelle listet jede verfügbare Vintage. Sichtbar wird dabei
auch der Bruch: Ab 2020 gelten die Grenzen des Zensus 2020.

In [ ]:
import re

session = requests.Session()
session.headers['User-Agent'] = dt.USER_AGENT

antwort = session.get(f'{dt.BASE_URL}/', timeout=30)
vintages = sorted({int(v) for v in re.findall(r'TIGER(20\d\d)/', antwort.text)})
print('Verfügbare TIGER-Vintages:')
print(' ', ', '.join(str(v) for v in vintages))
print()
print('Grenzen des Zensus 2010:', [v for v in vintages if v < 2020])
print('Grenzen des Zensus 2020:', [v for v in vintages if v >= 2020])

### 6.2 Die Auslieferungen selbst

`HEAD` genügt, um Grösse und Stand zu sehen, ohne die Dateien zu laden. Die
Quelle liefert kein `ETag`, aber ein `Last-Modified` - das hält das Manifest
fest.

In [ ]:
print(f'{"Vintage":<8} {"Staat":<10} {"Status":<7} {"Grösse":>10}  Stand')
for vintage in (2024, 2020):
    for fips, name in sorted(STATES.items()):
        url = dt.tiger_url(vintage, fips)
        kopf = session.head(url, timeout=30, allow_redirects=True)
        bytes_gross = int(kopf.headers.get('Content-Length', 0))
        print(f'{vintage:<8} {name:<10} {kopf.status_code:<7} '
              f'{bytes_gross/1e6:>7.1f} MB  {kopf.headers.get("Last-Modified", "-")}')

### 6.3 Die Missing-Key-Falle vorführen

Diese Zelle fragt die ACS-API **absichtlich ohne Schlüssel**. Zu sehen ist der
Kern des Problems: Status 200, Content-Type `text/html`, und ein Rumpf, der
nach einer normalen Seite aussieht. Danach greift die Prüfung der Ingestion.

In [ ]:
roh = requests.get(
    da.acs_url(da.DEFAULT_VINTAGE),
    params=da.query('26'),          # ohne key-Parameter
    timeout=60,
    headers={'User-Agent': da.USER_AGENT},
)
print('Status       :', roh.status_code, '  <- kein Fehlerstatus!')
print('Content-Type :', roh.headers.get('Content-Type'))
print('Rumpf        :', roh.text[:70].replace(chr(10), ' '), '...')
print()

try:
    da.parse_payload(roh.content)
except da.IngestionError as fehler:
    print('parse_payload() fängt es ab:')
    print('  ', fehler)

### 6.4 Die echte Abfrage

Mit Schlüssel aus `.env`. Der Schlüssel selbst wird nirgends ausgegeben.

In [ ]:
try:
    api_key = da.census_key()
    print('Schlüssel geladen, Länge:', len(api_key))
except da.IngestionError as fehler:
    api_key = None
    print('Kein Schlüssel:', fehler)

antwort = None
if api_key:
    antwort = da.fetch_table(session, da.DEFAULT_VINTAGE, '26', api_key)
    print('Spalten :', antwort.header)
    print('Zeilen  :', f'{len(antwort.rows):,}')
    print('Typen   :', [type(v).__name__ for v in antwort.rows[0]])
    print()
    print('Beispielzeile:')
    for name, wert in zip(antwort.header, antwort.rows[0]):
        print(f'  {name:14} {wert}')

### 6.5 Alles ist Text - und `-666666666` bleibt stehen

Zwei Feststellungen auf einmal: Die API sendet **jeden** Wert als
JSON-String, auch Zahlen. Und Tracts ohne belastbare Schätzung tragen den
Sentinel. Beide Eigenschaften übernimmt Bronze unverändert.

In [ ]:
if antwort is not None:
    i = antwort.header.index('B19013_001E')      # Median-Haushaltseinkommen
    p = antwort.header.index('B01003_001E')      # Einwohner
    sentinel = [r for r in antwort.rows if str(r[i]).startswith('-')]

    print(f'Tracts insgesamt           : {len(antwort.rows):,}')
    print(f"davon mit {da.MISSING_VALUE_CODE}    : {len(sentinel):,}")
    print(f'davon mit 0 Einwohnern     : {sum(1 for r in antwort.rows if r[p] == "0"):,}')
    print()
    print('Ein Tract mit Sentinel:')
    for name, wert in zip(antwort.header, sentinel[0]):
        print(f'  {name:14} {wert}')
    print()
    print('-> Bronze lässt den Wert stehen. Ihn als fehlend zu deuten,')
    print('   ist Aufgabe der Silver-Schicht.')

### 6.6 Der Beweis, dass der Join hält

Der entscheidende Punkt der ganzen Quelle: Passen die Tracts aus der
**Geometrie** (TIGER) und den **Attributen** (ACS) exakt zusammen? Beide
stammen vom selben Herausgeber und derselben Vintage, also müssen sie
deckungsgleich sein - aber nachgewiesen ist besser als angenommen.

Diese Prüfung ist fest in die Ingestion eingebaut: `download_acs` führt sie
bei jedem Lauf durch, sofern ein TIGER-Lauf derselben Vintage vorliegt, und
schreibt das Ergebnis als `completeness.tiger_cross_check` ins Manifest.

Die Zelle nutzt die bereits geschriebenen Bronze-Läufe. Fehlen sie, zeigt sie,
wie man sie erzeugt.

In [ ]:
import pyarrow.parquet as pq

DATA_ROOT = PROJECT_ROOT / 'data'
VINTAGE, FIPS = da.DEFAULT_VINTAGE, '26'

tiger_runs = dt.completed_runs(DATA_ROOT, VINTAGE, FIPS)
acs_runs = da.completed_runs(DATA_ROOT, VINTAGE, FIPS)

if tiger_runs and acs_runs:
    geometrie = set(pq.ParquetFile(tiger_runs[-1] / dt.PARQUET_FILE)
                    .read().column('GEOID').to_pylist())
    attribute = set(pq.ParquetFile(acs_runs[-1] / da.PARQUET_FILE)
                    .read().column('GEOID').to_pylist())

    print('Tracts in TIGER :', f'{len(geometrie):,}')
    print('Tracts in ACS   :', f'{len(attribute):,}')
    print('Schnittmenge    :', f'{len(geometrie & attribute):,}')
    print('nur in TIGER    :', sorted(geometrie - attribute)[:5] or '-')
    print('nur in ACS      :', sorted(attribute - geometrie)[:5] or '-')
    print()
    print('=> deckungsgleich:', geometrie == attribute)
else:
    print('Noch keine Bronze-Läufe vorhanden. Erzeugen mit:')
    print()
    print(f'  python -m census_pipeline.download_tiger --vintage {VINTAGE} --state {FIPS}')
    print(f'  python -m census_pipeline.download_acs   --vintage {VINTAGE} --state {FIPS}')

### 6.7 Wie nah liegen die Fälle an den Tract-Grenzen?

Die Behauptung aus 5.3 nachgemessen. Gefragt ist der Abstand jedes verorteten
Falles zur Grenze **des Tracts, dem er zugeordnet wurde**. Läge er zufällig im
Tract, müsste dieser Abstand breit streuen; der Vergleichswert dafür steht am
Ende der Zelle.

Die Zelle braucht die Silver-Tabellen `tract` und `crime_offense`.

In [ ]:
import duckdb

tracts = sorted((DATA_ROOT / 'silver' / 'dataset=tract').glob('**/*.parquet'))
faelle = sorted((DATA_ROOT / 'silver' / 'dataset=crime_offense').glob('**/*.parquet'))

if tracts and faelle:
    con = duckdb.connect()
    con.execute('INSTALL spatial; LOAD spatial;')
    con.execute('CREATE VIEW tr AS SELECT geoid, ST_GeomFromWKB(geometry) AS g '
                f'FROM read_parquet({[str(x) for x in tracts]})')
    con.execute('CREATE VIEW co AS SELECT city, tract_geoid AS geoid, longitude, latitude '
                f'FROM read_parquet({[str(x) for x in faelle]}) '
                'WHERE longitude IS NOT NULL AND tract_geoid IS NOT NULL')
    con.execute("""CREATE TABLE abstand AS
        SELECT co.city,
               ST_Distance(ST_Point(co.longitude, co.latitude), ST_Boundary(tr.g)) AS grad
        FROM co JOIN tr ON tr.geoid = co.geoid""")

    print('Abstand zur Grenze des eigenen Tracts (1 Grad Länge ~ 82,7 km):')
    for grad, text in [(1e-7, '1 cm'), (1e-6, '8 cm'), (1e-5, '~1 m'),
                       (1e-4, '~9 m'), (1e-3, '~90 m')]:
        anteil = con.execute('SELECT 100.0 * count(*) FILTER (WHERE grad < '
                             f'{grad}) / count(*) FROM abstand').fetchone()[0]
        print(f'   näher als {text:>6}: {anteil:6.2f} %')

    print()
    for stadt, n, nah in con.execute(
            'SELECT city, count(*), count(*) FILTER (WHERE grad < 1e-5) '
            'FROM abstand GROUP BY 1 ORDER BY 1').fetchall():
        print(f'   {stadt:9} {nah:>7,} von {n:>7,} näher als ~1 m  -> {100 * nah / n:5.2f} %')

    erwartet = con.execute("""
        SELECT sum(k.n * m.umfang / m.flaeche) / sum(k.n) * 100
        FROM (SELECT geoid, count(*) AS n FROM co GROUP BY 1) k
        JOIN (SELECT geoid, ST_Perimeter_Spheroid(g) AS umfang,
                     ST_Area_Spheroid(g) AS flaeche FROM tr) m ON m.geoid = k.geoid
    """).fetchone()[0]
    print()
    print(f'Erwartung bei Gleichverteilung im Tract: {erwartet:5.2f} % im 1-m-Randstreifen')
    print()
    print('-> Der Überhang ist systematisch, nicht zufällig: beide Städte setzen ihre')
    print('   Koordinaten auf Strassenachsen, und TIGER schneidet die Tract-Grenzen')
    print('   entlang derselben Strassen. Ein Versatz von ein bis zwei Metern - die')
    print('   Grösse des Unterschieds NAD83/WGS84 - entscheidet für diese Fälle über')
    print('   die Zuordnung.')
else:
    print('Keine Silver-Tabellen vorhanden. Erzeugen mit:')
    print()
    print('  python -m silver_pipeline.build_tract')
    print('  python -m silver_pipeline.build_crime_offense')


## 7. Datenqualität und Grenzen

**Der ACS schätzt, er zählt nicht.** Jeder Wert hat eine Fehlermarge, die
über die Variante `...M` statt `...E` abrufbar wäre. Auf Tract-Ebene sind
diese Margen beträchtlich - bei kleinen Tracts kann die Marge ein Drittel des
Schätzwerts erreichen. Für Einwohnerzahlen als Nenner ist das vertretbar, für
feine Aussagen zu Einkommensunterschieden nicht.

**Block Groups wären feiner, aber unsicherer.** Eine Ebene unter dem Tract
liegen die Block Groups. Ihre ACS-Werte sind so unsicher, dass sie für
Ratenbildung kaum taugen - deshalb arbeitet die Pipeline auf Tract-Ebene.

**Tracts ohne Einwohner gibt es wirklich.** Wasserflächen, Flughäfen,
Industriegebiete und Parks erscheinen als Tract mit Bevölkerung 0. Eine Rate
"Fälle pro 1'000 Einwohner" ist dort nicht definiert - der Nenner ist null.
Das muss die Silver-Schicht abfangen, sonst entstehen unendliche Raten.

**Zeitliche Passung.** Die Falldaten sind tagesaktuell, die ACS-Werte bilden
fünf Jahre ab. Eine Rate aus Fällen 2024 und ACS 2020-2024 ist üblich und
vertretbar, aber sie mischt zwei Zeitbegriffe. Das gehört in jede
Ergebnisdarstellung.

**Kein Personenbezug.** Beide Produkte sind aggregierte öffentliche Daten. Das
Census Bureau wendet auf die Zensus-Ergebnisse zusätzlich
Differential Privacy an, was einzelne kleine Werte leicht verrauschen kann.

## 8. Entscheidungen für die Census-Ingestion

| Entscheidung | Begründung |
| --- | --- |
| Eigene Pipeline, keine Kopie der Chicago-Ingestion | Referenzdaten statt Ereignisse: kein `updated_on`, also kein Wasserzeichen, kein Lookback, kein Delta |
| Statt Wasserzeichen die Frage "Vintage schon geladen?" | Eine Vintage ist eingefroren; ein zweiter Lauf brachte dieselben Bytes. `--force` erzwingt ihn trotzdem |
| Partition nach `vintage=` und `state=` | Das Abrufdatum ordnet hier nichts; es steckt weiterhin in der `run_id` |
| Nur Tracts, keine Block Groups | ACS-Werte auf Block-Group-Ebene sind für Ratenbildung zu unsicher |
| Zwei Auslieferungen: `tract` und `place` | Ein Tract sagt nicht, zu welcher Stadt er gehört. Cook County hat 5,18 Mio Einwohner, Chicago rund 2,7 Mio - ohne Stadtgrenze gibt es keinen Nenner |
| Geometrie als WKB im Parquet, via pyogrio | Liest Shapefiles direkt aus dem ZIP nach Arrow - geopandas und shapely braucht es dafür nicht |
| Gepinntes Schema plus `_extra` | Das Quellschema driftet zwischen Vintages (`GEOIDFQ`); so bleibt jeder Lauf gemeinsam lesbar |
| Spaltennamen bleiben ACS-Codes | `B01003_001E` statt `einwohner`: Bronze bleibt textgetreu, die Bedeutung hält das Manifest fest |
| `GEOID` als einzige abgeleitete Spalte | Ohne sie ist die Tabelle nicht joinbar, und jeder Konsument müsste dieselbe Verkettung nachbauen. Das Manifest weist sie als abgeleitet aus |
| Kreuzabgleich ACS gegen TIGER | Die Quelle nennt keine Zeilenzahl; die Deckung der Tract-Mengen ist der bestmögliche Nachweis |
| Schlüssel nie im Manifest | Er gehört bei dieser API in die Abfrage - die abgelegte URL wird deshalb ohne ihn gebildet |
| Kein `ST_Transform` beim Join | NAD83 und WGS84 liegen ein bis zwei Meter auseinander. Die Verortung auf Strassenachsen ist auf dieser Ebene ohnehin nicht belastbar (5.3, 6.7) - die Umrechnung bleibt bewusst offen, nicht übersehen |

Der räumliche Join selbst - Fall-Koordinate in Tract-Polygon - gehört
**nicht** hierher, sondern in die Silver-Schicht: In Bronze sind `latitude` und
`longitude` noch Text. Eine Umrechnung zwischen den Bezugssystemen findet dort
allerdings nicht statt; was das bedeutet, steht in 5.3.

## 9. Die Ingestion aufrufen

```bash
# Geometrien: Tracts und Stadtgrenzen, beide Bundesstaaten
python -m census_pipeline.download_tiger --vintage 2024
python -m census_pipeline.download_tiger --vintage 2024 --auslieferung place

# Attribute, beide Bundesstaaten
python -m census_pipeline.download_acs --vintage 2024

# Einzelner Staat, ältere Vintage, erneuter Lauf trotz vorhandener Daten
python -m census_pipeline.download_tiger --vintage 2020 --state 17 --force
```

`--auslieferung` hat die Vorgabe `tract`. Die Stadtgrenzen brauchen deshalb
einen eigenen Lauf; wer ihn vergisst, hat später keinen Nenner.

Alle Aufrufe schreiben Raw und Bronze unter einer gemeinsamen `run_id`, mit
Manifest je Schicht.

## 10. Offizielle Quellen

- TIGER/Line Shapefiles: <https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html>
- Verzeichnis der Auslieferungen: <https://www2.census.gov/geo/tiger/>
- Technische Dokumentation TIGER/Line: <https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html>
- ACS 5-Jahres-Schätzungen: <https://www.census.gov/data/developers/data-sets/acs-5year.html>
- Variablenliste ACS5 2024: <https://api.census.gov/data/2024/acs/acs5/variables.html>
- Schlüssel beantragen: <https://api.census.gov/data/key_signup.html>
- Erklärung der Census Tracts: <https://www.census.gov/programs-surveys/geography/about/glossary.html>

## Appendix A: Die gepinnten Schemata

Beide Module pinnen ihr Parquet-Schema, damit jeder Lauf gemeinsam lesbar
bleibt. Alles, was nicht hineinpasst, landet in `_extra`.

In [ ]:
for aus in dt.AUSLIEFERUNGEN.values():
    schema = aus.schema()
    print(f'TIGER {aus.name:6} -', len(schema), 'Spalten')
    for feld in schema:
        print(f'   {feld.name:14} {feld.type}')
    print()
print('ACS    -', len(da.SCHEMA), 'Spalten')
for feld in da.SCHEMA:
    label = da.VARIABLES.get(feld.name, '')
    print(f'   {feld.name:14} {str(feld.type):8} {label}')

## Appendix B: Die FIPS-Nummern

Die Nummern sind der Schlüssel der Quelle - sie stecken im Dateinamen der
Shapefiles, in der ACS-Abfrage und in den ersten Stellen jeder `GEOID`.

| Ebene | Chicago | Detroit |
| --- | --- | --- |
| Bundesstaat | Illinois `17` | Michigan `26` |
| County | Cook `031` | Wayne `163` |
| GEOID-Präfix | `17031...` | `26163...` |

Die Pipeline lädt jeweils den **ganzen Bundesstaat**. Das ist kaum teurer als
ein einzelnes County, hält aber die Tür offen für weitere Städte, und der
Zuschnitt auf die Stadtgrenze passiert ohnehin erst beim räumlichen Join.